[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/08_support-rag-assistant.ipynb)

        *Industry Project 8 of 8 · AI Engineer · Advanced · about 8–10 hours*

        # Customer-Support Assistant with RAG

        **The situation.** *ShopKart*'s support team answers the same questions all day: returns, refunds, delivery, payments. They want an assistant that answers customers **from the help-centre articles only**, cites the article it used, and says "I don't know" rather than inventing a policy.

That's **retrieval-augmented generation (RAG)**: find the right passages, then ask an LLM to answer using only them. Most RAG failures are retrieval failures — the model can't use a passage it was never given — so that's where you'll spend your effort, and you'll measure it properly with a labelled test set.

        **What you'll hand in:** a chunked knowledge base, a BM25 retriever written by hand, a retrieval evaluation (hit@1, hit@3, MRR), an improved retriever with before/after numbers, a grounded prompt with citations, an extractive fallback answerer, and (optionally) answers from Claude.

        **Skills:** RAG, chunking, BM25, retrieval evaluation, hit@k / MRR, query expansion, prompt design, grounded answers, Claude API

        **How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**kb_articles.csv** — 18 help-centre articles: doc_id, title, text (paragraphs separated by a blank line)

**kb_questions.csv** — 30 real-style customer questions with the article that answers them (doc_id): your **evaluation set**. Never tune on what you'll report — in a real project you'd keep part of it hidden.

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return False
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return False
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return True
    print(f"❌ {name}: not yet — {_why(got, want)}")
    return False

import re
from collections import Counter
kb = load("kb_articles.csv")
questions = load("kb_questions.csv")
STOP = set("a an the is are was were be to of and or in on at for with my i me you your do does did can how what when where who which it this that from by as if after before not no any have has get".split())
SYNONYMS = {"money": ["refund"], "back": ["return", "refund"], "shipping": ["delivery"], "ship": ["delivery"],
            "package": ["order", "tracking"], "parcel": ["order", "tracking"], "broken": ["damaged", "defective"],
            "close": ["delete", "deletion"], "debited": ["deducted", "payment"], "escalate": ["grievance", "complaint"],
            "phone": ["support", "call"], "care": ["support"], "tomorrow": ["express", "next", "day"],
            "headphones": ["electronics"], "laptop": ["electronics", "warranty"], "cream": ["beauty", "opened"],
            "otp": ["password"], "expire": ["valid"], "colour": ["wrong", "item"], "pickup": ["return", "courier"]}


def tokens(text):
    # lower-case words without stop words — used by every retriever
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]


# Used by later milestones and the checks (they call the functions you write).
def retrieve_better(question, k=3):
    chunks = chunk(kb)
    ids = make_better_search(chunks)(question, k)
    return chunks.set_index("chunk_id").loc[ids].reset_index()


def prompt_rules_ok(question):
    chunks = retrieve_better(question, 5)
    p = build_prompt(question, chunks, max_words=120)
    src = p.split("Sources:")[1].split("Customer question:")[0]
    return ["I don't know based on the help centre." in p, f"[{chunks.doc_id[0]}]" in src,
            len(src.split()) <= 120, p.rstrip().endswith(question)]
print("Ready! Data loaded.")

## Milestone 1: Chunk the knowledge base

LLMs need short, focused passages. Write `chunk(kb, max_words=30)` returning a DataFrame with columns `chunk_id`, `doc_id`, `title`, `text`:
- split each article into paragraphs on blank lines,
- if a paragraph has more than `max_words` words, split it into consecutive pieces of at most `max_words` words,
- `chunk_id` = `"<doc_id>-<n>"` with n counting from 1 within each article.

In [ ]:
def chunk(kb, max_words=30):
    raise NotImplementedError

In [ ]:
check("Chunk the knowledge base", lambda: [len(chunk(kb)), int(chunk(kb).groupby('doc_id').size().max()), chunk(kb)['chunk_id'].tolist()[:4]], [40, 3, ['KB01-1', 'KB01-2', 'KB02-1', 'KB02-2']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def chunk(kb, max_words=30):
    rows = []
    for doc_id, title, text in kb[["doc_id", "title", "text"]].itertuples(index=False):
        n = 0
        for para in text.split("\n\n"):
            words = para.split()
            for start in range(0, len(words), max_words):
                n += 1
                rows.append((f"{doc_id}-{n}", doc_id, title, " ".join(words[start:start + max_words])))
    return pd.DataFrame(rows, columns=["chunk_id", "doc_id", "title", "text"])
```

</details>

## Milestone 2: BM25 search, from scratch

**BM25** is the classic keyword ranking behind search engines, and still a strong first stage in RAG. For a query q and chunk d:

score(d) = Σ over query terms t of IDF(t) × f(t, d) × (k₁ + 1) / (f(t, d) + k₁ × (1 − b + b × |d| / avgdl))

with IDF(t) = ln(1 + (N − n(t) + 0.5) / (n(t) + 0.5)), f = term count in the chunk, |d| = chunk length in tokens, avgdl = average length, N = number of chunks, n(t) = chunks containing t, k₁ = 1.5, b = 0.75.

Write `make_bm25(chunks)` returning a function `search(query, k)` that returns the top-k `chunk_id`s (ties: earlier chunk first). Use `tokens()` for both chunks and queries.

<details>
<summary><b>💡 Hint</b></summary>

Precompute token Counters per chunk, the lengths, and document frequencies. Score in a loop, then sort by (−score, position).

</details>

In [ ]:
def make_bm25(chunks, k1=1.5, b=0.75):
    def search(query, k):
        raise NotImplementedError
    return search

In [ ]:
check("BM25 search, from scratch", lambda: [make_bm25(chunk(kb))('how many days do card refunds take', 3), make_bm25(chunk(kb))('is cash on delivery available', 2)], [['KB03-2', 'KB03-1', 'KB16-2'], ['KB08-2', 'KB08-1']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def make_bm25(chunks, k1=1.5, b=0.75):
    docs = [Counter(tokens(t)) for t in chunks["text"]]
    lengths = [sum(d.values()) for d in docs]
    avgdl = sum(lengths) / len(lengths)
    N = len(docs)
    df = Counter(t for d in docs for t in d)
    idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
    ids = chunks["chunk_id"].tolist()

    def search(query, k):
        q = tokens(query)
        scores = []
        for i, d in enumerate(docs):
            s = 0.0
            for t in q:
                f = d.get(t, 0)
                if f:
                    s += idf[t] * f * (k1 + 1) / (f + k1 * (1 - b + b * lengths[i] / avgdl))
            scores.append((-s, i))
        return [ids[i] for _, i in sorted(scores)[:k]]
    return search
```

</details>

## Milestone 3: Measure retrieval

Write `evaluate(search, questions, k=3)` returning `{"hit@1": %, "hit@k": %, "mrr": ...}`:
- a question is a **hit@k** if any of the top-k chunks comes from its correct `doc_id`,
- **MRR** = mean of 1 / (rank of the first chunk from the correct doc), 0 if not in the top k.

Percentages rounded to 1, MRR to 3. Then look at the questions it misses — why do they fail?

In [ ]:
def evaluate(search, questions, k=3):
    raise NotImplementedError

In [ ]:
check("Measure retrieval", lambda: evaluate(make_bm25(chunk(kb)), questions), {'hit@1': 56.7, 'hit@k': 76.7, 'mrr': 0.656}, tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def evaluate(search, questions, k=3):
    h1 = hk = rr = 0
    for q, doc in questions[["question", "doc_id"]].itertuples(index=False):
        docs = [c.split("-")[0] for c in search(q, k)]
        if docs and docs[0] == doc:
            h1 += 1
        if doc in docs:
            hk += 1
            rr += 1 / (docs.index(doc) + 1)
    n = len(questions)
    return {"hit@1": round(h1 / n * 100, 1), "hit@k": round(hk / n * 100, 1), "mrr": round(rr / n, 3)}
```

</details>

Try `make_bm25(chunk(kb))('When will I get my refund?', 3)`: the article says *refunds*, the query says *refund*, and plain BM25 treats them as different words. Real search engines **stem** words (refunds → refund) — a good extension. Look at the misses: customers say *"money back"*, *"broken"*, *"close my account"*; the articles say *refund*, *damaged*, *delete*. Keyword search can't bridge different words for the same idea. That's the **vocabulary mismatch** problem — the next milestone attacks it.

## Milestone 4: Make retrieval better

Two cheap, effective fixes:
1. **Index the title with every chunk** (search `title + " " + text`), so a chunk about refunds is findable even if the word appears only in the title.
2. **Expand the query** with `SYNONYMS`: add the listed words for every query token that has an entry.

Write `make_better_search(chunks)` returning `search(query, k)` that does both on top of your BM25. The check reports the new evaluation — compare with the previous milestone.

<details>
<summary><b>💡 Hint</b></summary>

Build a copy of `chunks` whose `text` is title + text and pass it to `make_bm25`. For the query: `query + ' ' + ' '.join(extra words)`.

</details>

In [ ]:
def make_better_search(chunks):
    def search(query, k):
        raise NotImplementedError
    return search

In [ ]:
check("Make retrieval better", lambda: evaluate(make_better_search(chunk(kb)), questions), {'hit@1': 80.0, 'hit@k': 96.7, 'mrr': 0.883}, tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def make_better_search(chunks):
    with_titles = chunks.assign(text=chunks["title"] + " " + chunks["text"])
    bm25 = make_bm25(with_titles)

    def search(query, k):
        extra = [s for t in tokens(query) for s in SYNONYMS.get(t, [])]
        return bm25(query + " " + " ".join(extra), k)
    return search
```

</details>

**Be honest about this number.** The synonym list was written while looking at these 30 questions, so it is tuned to them and the score is optimistic. Write 10 *new* questions and evaluate on those before you believe it. Production systems close the vocabulary gap with **embeddings** (dense vectors from a model) and combine them with BM25 (hybrid search), then re-rank. The synonym list is the hand-made version of the same idea — and you've now measured exactly how much it helps.

## Milestone 5: A grounded prompt with citations

Write `build_prompt(question, chunks, max_words=250)` returning one string that:
- tells the model to answer **only** from the sources, to cite the source id in square brackets like `[KB03]`, and to reply exactly `I don't know based on the help centre.` if the sources don't contain the answer,
- lists the sources as `[doc_id] title: text`, adding them in order and **stopping before the sources part would exceed `max_words` words**,
- ends with the customer's question.

The check verifies the rules on a real question (it doesn't compare exact wording).

In [ ]:
def build_prompt(question, chunks, max_words=250):
    raise NotImplementedError

In [ ]:
check("A grounded prompt with citations", lambda: prompt_rules_ok('How much is express shipping?'), [True, True, True, True], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def build_prompt(question, chunks, max_words=250):
    sources, used = [], 0
    for row in chunks.itertuples(index=False):
        line = f"[{row.doc_id}] {row.title}: {row.text}"
        if used + len(line.split()) > max_words:
            break
        sources.append(line)
        used += len(line.split())
    rules = ("You are ShopKart's support assistant. Answer the customer using ONLY the sources below. "
             "Cite the source id in square brackets, like [KB03], after each fact. "
             "If the sources do not contain the answer, reply exactly: I don't know based on the help centre.")
    return rules + "\n\nSources:\n" + "\n".join(sources) + f"\n\nCustomer question: {question}"
```

</details>

## Milestone 6: An answer without an LLM: extractive fallback

When the LLM is down (or too expensive for simple questions), return the single best **sentence** from the retrieved chunks. Write `extractive_answer(question, k=3)` that:
- retrieves the top-k chunks with `retrieve_better(question, k)` (in the setup cell; it uses your `chunk` and `make_better_search`),
- splits them into sentences (on `. ` / `.`),
- returns `(best sentence, its doc_id)`, where best = most query tokens shared, after synonym expansion (ties: earlier sentence first).

In [ ]:
def extractive_answer(question, k=3):
    raise NotImplementedError

In [ ]:
check("An answer without an LLM: extractive fallback", lambda: [extractive_answer(q)[1] for q in ['When will I get my money back on my credit card?', 'Is shipping free?', 'Someone called asking for my OTP']], ['KB03', 'KB05', 'KB14'], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def extractive_answer(question, k=3):
    q = set(tokens(question)) | {s for t in tokens(question) for s in SYNONYMS.get(t, [])}
    best = None
    for row in retrieve_better(question, k).itertuples(index=False):
        for sentence in re.split(r"(?<=\.)\s+", row.text):
            overlap = len(q & set(tokens(sentence)))
            if best is None or overlap > best[0]:
                best = (overlap, sentence.strip(), row.doc_id)
    return best[1], best[2]
```

</details>

Print the sentences too: `extractive_answer('Is shipping free?')`. Extractive answers are always grounded (they *are* the source), but can't combine facts or rephrase. That's what the LLM adds — below.

## Generate the answer with Claude (optional)

With retrieval and the prompt done, the LLM call is a few lines. It needs an API key and costs a fraction of a rupee per question, so it's optional. Compare its answers with your extractive ones: does it cite sources? Does it say "I don't know" for a question the help centre doesn't cover, like *"Do you sell cars?"*

In [ ]:
# Optional: generate the final answer with Claude.
# Needs an Anthropic API key (console.anthropic.com) — this costs a little money per call.
#   pip install anthropic   and set the ANTHROPIC_API_KEY environment variable (in Colab: Secrets).
import os, importlib.util
if os.environ.get("ANTHROPIC_API_KEY") and importlib.util.find_spec("anthropic"):
    import anthropic
    client = anthropic.Anthropic()
    question = "Can I return headphones that work fine?"
    prompt = build_prompt(question, retrieve_better(question, 3))
    message = client.messages.create(
        model="claude-haiku-5-5",        # fast and cheap; try "claude-opus-5-5" for the strongest answers
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    print(message.content[0].text)
else:
    print("No API key or anthropic package — skipping the LLM call. Your retrieval and prompt are the hard part, and they're done.")

## Present your work

Write a short design doc for the support team:

1. **Architecture**: chunking → BM25 (+ titles, synonyms) → prompt with citations → LLM, with the extractive fallback.
2. **Retrieval results**: hit@1, hit@3 and MRR before and after your improvements, and 2–3 examples of remaining failures.
3. **Safety**: grounding rules, citations, the "I don't know" behaviour, and why the bot must never invent policy.
4. **Cost and latency**: an LLM call only when needed; extractive answers for simple questions.
5. **Next steps**: embeddings / hybrid search, re-ranking, an answer-quality evaluation, feedback buttons.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built a retrieval-augmented support assistant over an 18-article help centre with hand-written BM25, title indexing and query expansion
- Created a 30-question labelled evaluation set and raised hit@3 from **X%** to **Y%** (MRR **A → B**)
- Designed a grounded prompt with citations and an "I don't know" rule, plus an extractive fallback, and generated answers with the Claude API

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. Why did you evaluate retrieval separately from the final answer?
2. Explain BM25 in plain words. What do k₁ and b control?
3. What is the vocabulary-mismatch problem, and how do embeddings solve it?
4. How do you stop the model from inventing a refund policy?
5. How would you chunk long documents, and how does chunk size change results?
6. How would you evaluate the final answers, not just retrieval?

## Take it further

- Add dense retrieval with a sentence-embedding model and combine it with BM25 (reciprocal rank fusion).
- Write 10 questions the help centre can't answer and measure how often the bot correctly says "I don't know".
- Use an LLM as a judge to score answer faithfulness against the sources.
- Serve the assistant as an API (project 7 shows how) with a chat front end.